# FrameWeave Hybrid v6.0 (V5.1) — NExT-GQA eval (Video-LLaVA-7B)

Notebook **chỉ eval**. Không train. So sánh **Vanilla vs Hybrid** trên **cùng K=8 frame**. Không DTR.

Hai giao thức: **oracle evidence** (chính) và **uniform full video**. Nhãn `gsub_*.json` chỉ dùng để chọn frame oracle. Không nhét span / đáp án đúng vào prompt. Số oracle **không** phải Acc@GQA.

```
outputs_reljoin_v6_videollava/
  checkpoints/best_external_val/Hybrid.pt
  eval_nextgqa/
```

Tải dữ liệu thẳng Hugging Face `shuzhig/nextgqa` revision `3fc4ace54d7566f46145ec29138e912e557ae870` vào `/content/nextgqa` (như HERBench lite). CSV persist Drive.

## Source code

Mở notebook này từ GitHub hoặc Colab. Run All clone [FrameWeave](https://github.com/nqMinh-GR/FrameWeave.git) vào `/content/FrameWeave`, rồi symlink `reljoin/` và `run_*.py` vào `/content`. Không upload file source.

Checkpoint và folder output train không nằm trong git. Lấy chúng từ `/content` hoặc Drive.

## Hardware

Colab **A100 40 GB**. Disk trống ≥ ~70 GB (model ~14 GB + NExT-GQA videos). Mount Drive để resume.

`SMOKE_FIRST=True` lần đầu. Sau smoke đặt `False` rồi `RUN_FULL=True` trên **val / oracle**. Khóa K, pixel, prompt. Sau đó uniform và test với cùng cấu hình. Không tune trên test.


## 1. Configuration


In [ ]:
# 1. Eval-only config — FrameWeave v6.0 (V5.1) / Video-LLaVA-7B / NExT-GQA
from __future__ import annotations

from pathlib import Path

assert Path("/content").exists(), "Notebook này chỉ chạy trên Google Colab (/content)."

ROOT = Path("/content")
SMOKE_FIRST = True
RUN_FULL = True
EVAL_SPLIT = "val"          # val trước; sau khi khóa cấu hình mới chuyển test
EVAL_MODE = "oracle"        # primary; "uniform" hoặc "both" sau khi oracle ổn
NUM_FRAMES = 8
METHOD_VERSION = "6.0.0"

VARIANT_TO_METHOD = {"Hybrid": "hybrid"}
REQUIRED_CKPTS = ["Hybrid.pt"]
METHODS = "vanilla,hybrid"

EXTRACT_DIR_CANDIDATES = [
    ROOT / "outputs_reljoin_v6_videollava",
    ROOT / "outputs_reljoin_v6_videollava (2)",
    Path("/content/drive/MyDrive/outputs_reljoin_v6_videollava"),
    Path("/content/drive/MyDrive/reljoin_v6_videollava_outputs") / "outputs_reljoin_v6_videollava",
    Path("/content/drive/MyDrive/reljoin_v6_videollava_outputs"),
]

MODEL_ID = "LanguageBind/Video-LLaVA-7B"
MODEL_REVISION = "aecae02b7dee5c249e096dcb0ce546eb6f811806"
NEXTGQA_ID = "shuzhig/nextgqa"
NEXTGQA_REVISION = "3fc4ace54d7566f46145ec29138e912e557ae870"

VENV = ROOT / "dtr_env"
PYTHON = VENV / "bin" / "python"
OUT_DIR = ROOT / "outputs_reljoin_v6_videollava"
CKPT_DIR = OUT_DIR / "checkpoints" / "best_external_val"
PRED_DIR = OUT_DIR / "eval_nextgqa"
MODEL_DIR = ROOT / "checkpoints" / "video-llava-7b"
NEXTGQA_ROOT = ROOT / "nextgqa"

print({
    "SMOKE_FIRST": SMOKE_FIRST,
    "RUN_FULL": RUN_FULL,
    "EVAL_SPLIT": EVAL_SPLIT,
    "EVAL_MODE": EVAL_MODE,
    "NUM_FRAMES": NUM_FRAMES,
    "METHODS": METHODS,
    "CKPT_DIR": str(CKPT_DIR),
    "NEXTGQA_ROOT": str(NEXTGQA_ROOT),
    "method": "FrameWeave-Hybrid",
    "method_version": METHOD_VERSION,
    "backbone": MODEL_ID,
    "not_acc_gqa": True,
})


## 2. Environment (`dtr_env`)

Python 3.11 + torch 2.0.1+cu118 — cùng pin với notebook DTR.


In [ ]:
# 2. Bootstrap dtr_env (Python 3.11 + torch 2.0.1 như DTR)
from __future__ import annotations

import os
import subprocess
from pathlib import Path

assert Path("/content").exists(), "This notebook targets Google Colab (/content)."
ROOT = Path("/content")
VENV = ROOT / "dtr_env"
VENV_BIN = VENV / "bin"
PYTHON = VENV_BIN / "python"
PIP = VENV_BIN / "pip"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def sh(cmd, check=True):
    print("$", " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), check=check)


sh(["apt-get", "update", "-qq"], check=False)
sh(["apt-get", "install", "-y", "-qq", "python3.11", "python3.11-venv", "python3.11-dev", "ffmpeg"], check=False)
py311 = Path("/usr/bin/python3.11")
assert py311.exists(), "Không tìm thấy python3.11 — kiểm tra apt install."

need_recreate = True
if PYTHON.exists():
    ver = subprocess.check_output([str(PYTHON), "-V"], text=True).strip()
    print("Existing venv:", ver)
    need_recreate = "3.11" not in ver
if need_recreate:
    if VENV.exists():
        sh(["rm", "-rf", str(VENV)])
    sh([str(py311), "-m", "venv", "--copies", str(VENV)])
if not PIP.exists():
    sh([str(PYTHON), "-m", "ensurepip", "--upgrade"], check=False)
if not PIP.exists():
    get_pip = Path("/tmp/get-pip.py")
    sh(["curl", "-sSL", "https://bootstrap.pypa.io/get-pip.py", "-o", str(get_pip)])
    sh([str(PYTHON), str(get_pip)])
sh([str(PIP), "install", "-q", "--upgrade", "pip", "setuptools<70", "wheel"])
print("Python in venv:", subprocess.check_output([str(PYTHON), "-V"], text=True).strip())
sh([
    str(PIP), "install", "--disable-pip-version-check",
    "torch==2.0.1+cu118", "torchvision==0.15.2+cu118",
    "--index-url", "https://download.pytorch.org/whl/cu118",
])
sh([
    str(PIP), "install", "--disable-pip-version-check", "--upgrade-strategy", "only-if-needed",
    "hf-transfer", "numpy==1.26.4", "transformers==4.31.0", "tokenizers==0.13.3", "accelerate==0.21.0",
    "datasets==2.21.0", "huggingface_hub==0.24.7", "einops==0.7.0", "sentencepiece==0.2.0", "protobuf<5",
    "av==12.3.0", "pandas==2.2.2", "scipy==1.13.1", "tqdm==4.67.1", "opencv-python-headless==4.10.0.84",
    "Pillow==10.4.0", "pytorchvideo==0.1.5", "fvcore==0.1.5.post20221221", "iopath==0.1.10", "decord==0.6.0",
])
check = subprocess.check_output([
    str(PYTHON), "-c",
    "import torch, torchvision, transformers; "
    "print(torch.__version__); print(torchvision.__version__); print(transformers.__version__); "
    "print(torch.__file__); print(torchvision.__file__); print(transformers.__file__)",
], text=True)
print(check)
for line in check.splitlines():
    if "/site-packages/" in line:
        assert "/content/dtr_env/" in line, f"Package leaked outside venv: {line}"
assert (VENV_BIN / "huggingface-cli").exists()
print("dtr_env ready:", PYTHON)


In [ ]:
!/content/dtr_env/bin/pip install --disable-pip-version-check "peft==0.4.0" "timm==0.6.13" "scikit-learn==1.3.2" "matplotlib==3.8.4"


## 3. Drive + pin DTR loader

DTR git là loader Video-LLaVA. Method `dtr` không chạy.


In [ ]:
# 3. Drive + clone DTR (Video-LLaVA loader)
from __future__ import annotations

import hashlib
import os
import subprocess
from pathlib import Path

ROOT = Path("/content")
VENV = ROOT / "dtr_env"
PYTHON = VENV / "bin" / "python"
DTR_REPO = ROOT / "DTR"
DTR_COMMIT = "b7e1ead192fe04502db70f69d3baebdf5d88d64e"
DTR_SOURCE_SHA256 = "01a62136185be235d6fcc135fa67f30934687f20b2c6cae59d4b948acbd54d1b"
OUT_DIR = ROOT / "outputs_reljoin_v6_videollava"
PRED_DIR = OUT_DIR / "eval_nextgqa"
MODEL_DIR = ROOT / "checkpoints" / "video-llava-7b"
for p in [MODEL_DIR.parent, OUT_DIR, PRED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    RELJOIN_PERSIST = Path("/content/drive/MyDrive/reljoin_v6_videollava_outputs")
except Exception as exc:
    print("Drive mount skipped:", exc)
    RELJOIN_PERSIST = OUT_DIR / "persist_local"
if RELJOIN_PERSIST.resolve() == OUT_DIR.resolve():
    RELJOIN_PERSIST = OUT_DIR / "persist_local"
    print("Persist dir was out_dir; using", RELJOIN_PERSIST)
RELJOIN_PERSIST.mkdir(parents=True, exist_ok=True)
os.environ["RELJOIN_PERSIST"] = str(RELJOIN_PERSIST)
os.environ["RELJOIN_ROOT"] = str(ROOT)

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

if not (DTR_REPO / ".git").exists():
    subprocess.run(["git", "clone", "https://github.com/ifwub234/DTR.git", str(DTR_REPO)], check=True)
subprocess.run(["git", "-C", str(DTR_REPO), "fetch", "--all"], check=False)
subprocess.run(["git", "-C", str(DTR_REPO), "checkout", DTR_COMMIT], check=True)
got = sha256_file(DTR_REPO / "run_inference_dtr.py")
assert got == DTR_SOURCE_SHA256, f"DTR SHA mismatch: {got}"
print({"DTR_REPO": str(DTR_REPO), "RELJOIN_PERSIST": str(RELJOIN_PERSIST), "OUT_DIR": str(OUT_DIR), "PRED_DIR": str(PRED_DIR)})


## 4. Clone source and verify


In [ ]:
# Fetch FrameWeave source. Run All clones the repo; do not upload .py files.
from __future__ import annotations

import subprocess
from pathlib import Path

REPO_URL = "https://github.com/nqMinh-GR/FrameWeave.git"
REPO_DIR = Path("/content/FrameWeave")
ROOT = Path("/content")
assert ROOT.exists(), "This notebook targets Google Colab (/content)."

if not (REPO_DIR / ".git").exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])


def link(src: Path, dest: Path) -> None:
    src = src.resolve()
    if dest.is_symlink():
        if dest.resolve() != src:
            dest.unlink()
            dest.symlink_to(src)
        return
    if dest.exists():
        raise RuntimeError(
            f"{dest} already exists and is not the repo link. Restart the runtime, then Run All."
        )
    dest.symlink_to(src)


CODE = REPO_DIR / "videollava"
link(CODE / "reljoin", ROOT / "reljoin")
for script in sorted(CODE.glob("run_*.py")):
    link(script, ROOT / script.name)
print("FrameWeave source linked from", CODE)


In [ ]:
# 4. Verify FrameWeave Hybrid NExT-GQA sources
from pathlib import Path

ROOT = Path("/content")
required = [
    ROOT / "run_nextgqa_reljoin.py",
    ROOT / "run_herbench_oracle_dtr.py",
    ROOT / "run_reljoin.py",
    ROOT / "reljoin" / "__init__.py",
    ROOT / "reljoin" / "config.py",
    ROOT / "reljoin" / "train.py",
    ROOT / "reljoin" / "reljoin_modules.py",
    ROOT / "reljoin" / "videollava_hooks.py",
    ROOT / "reljoin" / "nextgqa.py",
    ROOT / "reljoin" / "nextgqa_frames.py",
    ROOT / "reljoin" / "diagnostics.py",
    ROOT / "reljoin" / "frame_io.py",
]
missing = [str(p) for p in required if not p.exists()]
assert not missing, "Missing files — clone or symlink failed:\n" + "\n".join(missing)
print("All FrameWeave Hybrid NExT-GQA sources present.")
for p in required:
    print(f"  OK {p} ({p.stat().st_size} bytes)")


## 5. Gắn checkpoint (`outputs_reljoin_v6_videollava/checkpoints`)

Bắt buộc `Hybrid.pt`. Checkpoint v3/v4/v5 **không** load vào v6.0.


In [ ]:
# 5. Bind extract checkpoints — same layout as train.py / make_output_tree
from __future__ import annotations

import json
import shutil
import zipfile
from pathlib import Path

ROOT = Path("/content")
OUT_DIR = ROOT / "outputs_reljoin_v6_videollava"
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "predictions").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "eval_nextgqa").mkdir(parents=True, exist_ok=True)

try:
    REQUIRED_CKPTS
    EXTRACT_DIR_CANDIDATES
    VARIANT_TO_METHOD
except NameError:
    REQUIRED_CKPTS = ["Hybrid.pt"]
    VARIANT_TO_METHOD = {"Hybrid": "hybrid"}
    EXTRACT_DIR_CANDIDATES = [
        ROOT / "outputs_reljoin_v6_videollava",
        ROOT / "outputs_reljoin_v6_videollava (2)",
        Path("/content/drive/MyDrive/outputs_reljoin_v6_videollava"),
        Path("/content/drive/MyDrive/reljoin_v6_videollava_outputs") / "outputs_reljoin_v6_videollava",
        Path("/content/drive/MyDrive/reljoin_v6_videollava_outputs"),
    ]

v2_pt = ROOT / "outputs_reljoin_residual" / "checkpoints" / "best_external_val" / "RelJoin.pt"
if v2_pt.is_file():
    print("NOTE: found v2 RelJoin.pt at", v2_pt, "— không dùng cho v6.0")


def ckpt_complete(best_dir: Path) -> bool:
    return best_dir.is_dir() and all((best_dir / n).is_file() for n in REQUIRED_CKPTS)


def match_tree(cand: Path) -> Path | None:
    if not cand.is_dir():
        return None
    best = cand / "checkpoints" / "best_external_val"
    last = cand / "checkpoints" / "last"
    if ckpt_complete(best):
        return cand
    if ckpt_complete(last):
        print("WARN: using checkpoints/last (best_external_val incomplete) at", cand)
        return cand
    if (cand / "Hybrid.pt").is_file():
        return cand
    return None


def find_extract_root() -> Path:
    for cand in EXTRACT_DIR_CANDIDATES:
        hit = match_tree(cand)
        if hit is not None:
            return hit
    zip_roots = [
        ROOT,
        Path("/content/drive/MyDrive"),
        Path("/content/drive/MyDrive/reljoin_v6_videollava_outputs"),
    ]
    for base in zip_roots:
        if not base.exists():
            continue
        for zpath in list(base.glob("*reljoin*.zip")) + list(base.glob("*output*.zip")):
            if not zpath.is_file():
                continue
            dest = Path("/tmp/vast_outputs_reljoin_v6_videollava")
            if dest.exists():
                shutil.rmtree(dest)
            print("Extracting", zpath)
            with zipfile.ZipFile(zpath) as z:
                z.extractall(dest)
            seen = []
            for cand in [dest, *dest.rglob("*")]:
                if not cand.is_dir() or cand in seen:
                    continue
                seen.append(cand)
                hit = match_tree(cand)
                if hit is not None:
                    print("Found checkpoint tree", hit)
                    return hit
            pts = [str(p.relative_to(dest)) for p in dest.rglob("*.pt")]
            print("Zip has no Hybrid.pt layout. *.pt:", pts[:20] or "none")
    scanned = "\n".join(
        f"  {c} exists={c.exists()} best={list((c / 'checkpoints' / 'best_external_val').glob('*.pt')) if (c / 'checkpoints' / 'best_external_val').exists() else 'NO DIR'}"
        for c in EXTRACT_DIR_CANDIDATES
    )
    raise FileNotFoundError(
        "Không thấy Hybrid.pt đúng layout train:\n"
        "  {extract}/checkpoints/best_external_val/Hybrid.pt\n"
        "Upload cả folder `outputs_reljoin_v6_videollava` (kèm checkpoints/) lên /content hoặc Drive.\n"
        f"Đã quét:\n{scanned}"
    )


EXTRACT_DIR = find_extract_root()
BEST_DIR = EXTRACT_DIR / "checkpoints" / "best_external_val"
LAST_DIR = EXTRACT_DIR / "checkpoints" / "last"
if ckpt_complete(BEST_DIR):
    CKPT_DIR = BEST_DIR
elif ckpt_complete(LAST_DIR):
    CKPT_DIR = LAST_DIR
else:
    reljoin_loose = EXTRACT_DIR / "Hybrid.pt"
    CKPT_DIR = EXTRACT_DIR if reljoin_loose.is_file() else BEST_DIR
CKPT_DIR.mkdir(parents=True, exist_ok=True)

resolved = {}
for name in REQUIRED_CKPTS:
    src = CKPT_DIR / name
    if not src.is_file():
        alt = LAST_DIR / name
        if alt.is_file():
            print(f"best missing {name} → copy from last/")
            shutil.copy2(alt, CKPT_DIR / name)
            src = CKPT_DIR / name
        else:
            resolved[name] = None
            continue
    mb = src.stat().st_size / 1e6
    print(f"OK {name}: {src} ({mb:.1f} MB)")
    resolved[name] = src

missing = [n for n, p in resolved.items() if p is None]
assert not missing, f"Thiếu checkpoint: {missing} dưới {CKPT_DIR} và {LAST_DIR}"

if CKPT_DIR.resolve() != LAST_DIR.resolve():
    for variant in VARIANT_TO_METHOD:
        name = f"{variant}.pt"
        if not (CKPT_DIR / name).is_file() and (LAST_DIR / name).is_file():
            print(f"missing {name} → copy from last/")
            shutil.copy2(LAST_DIR / name, CKPT_DIR / name)

dst_best = OUT_DIR / "checkpoints" / "best_external_val"
if CKPT_DIR.resolve() != dst_best.resolve():
    dst_best.mkdir(parents=True, exist_ok=True)
    for src in CKPT_DIR.glob("*.pt"):
        shutil.copy2(src, dst_best / src.name)
        print("Copied", src.name, "->", dst_best / src.name)
    CKPT_DIR = dst_best

found_methods = ["vanilla"]
for variant, method in VARIANT_TO_METHOD.items():
    path = CKPT_DIR / f"{variant}.pt"
    if path.is_file():
        if path.name not in REQUIRED_CKPTS:
            mb = path.stat().st_size / 1e6
            print(f"OK {path.name}: {path} ({mb:.1f} MB)")
        found_methods.append(method)
METHODS = ",".join(found_methods)

for aux in ["train_summary.json", "config.json", "environment.txt"]:
    src = EXTRACT_DIR / aux
    if src.exists() and src.resolve() != (OUT_DIR / aux).resolve():
        shutil.copy2(src, OUT_DIR / aux)
        print("Copied", aux)
log_src = EXTRACT_DIR / "logs" / "train_metrics.csv"
if log_src.exists():
    (OUT_DIR / "logs").mkdir(parents=True, exist_ok=True)
    dst = OUT_DIR / "logs" / "train_metrics.csv"
    if log_src.resolve() != dst.resolve():
        shutil.copy2(log_src, dst)

summary_path = EXTRACT_DIR / "train_summary.json"
if summary_path.exists():
    print("train_summary:", json.loads(summary_path.read_text()))

print({
    "EXTRACT_DIR": str(EXTRACT_DIR),
    "CKPT_DIR": str(CKPT_DIR),
    "METHODS": METHODS,
    "pts": sorted(p.name for p in CKPT_DIR.glob("*.pt")),
})


## 6. Tải backbone + NExT-GQA (`shuzhig/nextgqa`)

Snapshot thẳng `/content`. Cell này bắt buộc trước smoke.


In [ ]:
# 6. Download Video-LLaVA-7B + shuzhig/nextgqa
from __future__ import annotations

import os
import shutil
import subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "dtr_env" / "bin" / "python"
assert PYTHON.exists(), "Chạy cell 2 trước"
os.environ["PATH"] = f"{ROOT / 'dtr_env' / 'bin'}:{os.environ.get('PATH', '')}"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

NEXTGQA_ROOT = ROOT / "nextgqa"
MODEL_DIR = ROOT / "checkpoints" / "video-llava-7b"
NEXTGQA_REVISION = "3fc4ace54d7566f46145ec29138e912e557ae870"
MODEL_HF_ID = "LanguageBind/Video-LLaVA-7B"
MODEL_REVISION = "aecae02b7dee5c249e096dcb0ce546eb6f811806"
for p in [NEXTGQA_ROOT, MODEL_DIR.parent]:
    p.mkdir(parents=True, exist_ok=True)

print("Downloading Video-LLaVA-7B...")
subprocess.run([str(PYTHON), "-c", f"""
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='{MODEL_HF_ID}',
    revision='{MODEL_REVISION}',
    local_dir=r'{MODEL_DIR}',
    ignore_patterns=['*.msgpack', '*.h5', 'flax_model*', 'tf_model*'],
)
print('model OK')
"""], check=True)
assert (MODEL_DIR / "config.json").exists(), f"Missing {MODEL_DIR / 'config.json'}"

print("Downloading NExT-GQA (annotations + videos)...")
subprocess.run([
    str(PYTHON), "-u", str(ROOT / "run_nextgqa_reljoin.py"), "prepare",
    "--data-root", str(NEXTGQA_ROOT), "--dataset-revision", NEXTGQA_REVISION,
], check=True)

n_mp4 = len(list((NEXTGQA_ROOT / "videos").glob("*.mp4"))) if (NEXTGQA_ROOT / "videos").is_dir() else 0
print({
    "videos_mp4": n_mp4,
    "val_csv": (NEXTGQA_ROOT / "annotations" / "val.csv").is_file(),
    "test_csv": (NEXTGQA_ROOT / "annotations" / "test.csv").is_file(),
    "model_config": (MODEL_DIR / "config.json").exists(),
    "free_GiB": round(shutil.disk_usage(ROOT).free / 2**30, 2),
})
assert (MODEL_DIR / "config.json").exists(), "model config.json missing"
assert (NEXTGQA_ROOT / "annotations" / "val.csv").is_file(), "val.csv missing"
assert n_mp4 >= 2, "NExT-GQA videos missing"
subprocess.run([
    str(PYTHON), "-u", str(ROOT / "run_nextgqa_reljoin.py"), "audit",
    "--data-root", str(NEXTGQA_ROOT),
    "--out-dir", str(ROOT / "outputs_reljoin_v6_videollava" / "eval_nextgqa"),
], check=True)
print("Downloads OK")


## 7. Smoke rồi eval

Smoke in path, `(video_id, qid)`, gold, spans, 8 timestamp **uniform và oracle**. Vanilla và Hybrid phải cùng frame hash.

Eval chạy **process tách**. Log: `/content/eval_nextgqa_smoke.log` / `/content/eval_nextgqa_full.log`.

Nếu job cũ treo: `!pkill -f run_nextgqa_reljoin` rồi xóa `/content/eval_nextgqa.job.json`. Sau smoke OK đặt `SMOKE_FIRST = False`. Val oracle trước; không tune trên test.

Gọi trực tiếp `run_nextgqa_reljoin.py`.


In [ ]:
# 7. NExT-GQA eval — smoke then full (NO training)
from __future__ import annotations

import json
import os
import shutil
import signal
import subprocess
import threading
import time
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "dtr_env" / "bin" / "python"
RUNNER = ROOT / "run_nextgqa_reljoin.py"
NEXTGQA_ROOT = ROOT / "nextgqa"
MODEL_DIR = ROOT / "checkpoints" / "video-llava-7b"
OUT_DIR = ROOT / "outputs_reljoin_v6_videollava"
PRED_DIR = OUT_DIR / "eval_nextgqa"
CKPT = CKPT_DIR if "CKPT_DIR" in dir() else (OUT_DIR / "checkpoints" / "best_external_val")
PRED_DIR.mkdir(parents=True, exist_ok=True)
JOB_FILE = ROOT / "eval_nextgqa.job.json"
DRIVE_DIR = Path("/content/drive/MyDrive/reljoin_v6_videollava_outputs") / "eval_nextgqa"


def _from_cell1(name, default):
    try:
        ip = get_ipython()
    except NameError:
        ip = None
    ns = ip.user_ns if ip is not None else {}
    return ns[name] if name in ns else default

SMOKE_FIRST = bool(_from_cell1("SMOKE_FIRST", True))
RUN_FULL = bool(_from_cell1("RUN_FULL", True))
METHODS = str(_from_cell1("METHODS", "vanilla,hybrid") or "vanilla,hybrid")
EVAL_SPLIT = str(_from_cell1("EVAL_SPLIT", "val"))
EVAL_MODE = str(_from_cell1("EVAL_MODE", "oracle"))
NUM_FRAMES = int(_from_cell1("NUM_FRAMES", 8))
NEXTGQA_REVISION = "3fc4ace54d7566f46145ec29138e912e557ae870"
MODEL_REVISION = "aecae02b7dee5c249e096dcb0ce546eb6f811806"

needed = list(REQUIRED_CKPTS) if "REQUIRED_CKPTS" in dir() else ["Hybrid.pt"]
missing = [n for n in needed if not (CKPT / n).exists()]
assert not missing, f"Chạy cell 5 trước. Thiếu {missing} trong {CKPT}."
assert (MODEL_DIR / "config.json").exists(), "Chạy cell 6 trước — model missing."
assert PYTHON.exists(), "Chạy cell 2 trước."
assert RUNNER.is_file(), f"Missing {RUNNER}"
n_mp4 = len(list((NEXTGQA_ROOT / "videos").glob("*.mp4"))) if (NEXTGQA_ROOT / "videos").exists() else 0
assert n_mp4 >= 2, f"Chạy cell 6 trước — {n_mp4} mp4."
print({"Hybrid.pt": str(CKPT / "Hybrid.pt"), "videos_mp4": n_mp4, "split": EVAL_SPLIT, "mode": EVAL_MODE, "K": NUM_FRAMES})

base_cmd = [
    str(PYTHON), "-u", str(RUNNER),
    "--root", "/content",
    "--data-root", str(NEXTGQA_ROOT),
    "--model-dir", str(MODEL_DIR),
    "--out-dir", str(PRED_DIR),
    "--persist-dir", str(PRED_DIR),
    "--dataset-revision", NEXTGQA_REVISION,
    "--model-revision", MODEL_REVISION,
    "--methods", METHODS,
    "--reljoin-ckpt", str(CKPT),
    "--dtr-repo", str(ROOT / "DTR"),
    "--dtr-sha256", "01a62136185be235d6fcc135fa67f30934687f20b2c6cae59d4b948acbd54d1b",
    "--venv-site", str(ROOT / "dtr_env" / "lib"),
    "--split", EVAL_SPLIT,
    "--mode", EVAL_MODE,
    "--num-frames", str(NUM_FRAMES),
]


def eval_env():
    env = os.environ.copy()
    env.pop("PYTHONPATH", None)
    env.pop("PYTHONHOME", None)
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONNOUSERSITE"] = "1"
    env["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
    return env


def best_effort_copy(src: Path, dst: Path, timeout: float = 8.0) -> bool:
    src, dst = Path(src), Path(dst)
    if not src.exists():
        return False
    try:
        if dst.exists() and src.resolve() == dst.resolve():
            return True
    except OSError:
        return False
    try:
        dst.parent.mkdir(parents=True, exist_ok=True)
    except OSError as exc:
        print(f"WARN: mkdir {dst.parent}: {exc!r}", flush=True)
        return False
    err = []

    def _copy():
        try:
            tmp = dst.with_name(dst.name + ".tmp")
            shutil.copy2(src, tmp)
            tmp.replace(dst)
        except BaseException as exc:
            err.append(exc)

    t = threading.Thread(target=_copy, daemon=True)
    t.start()
    t.join(timeout)
    if t.is_alive() or err:
        print(f"WARN: copy issue {src} -> {dst}", flush=True)
        return False
    return True


def push_drive():
    if not DRIVE_DIR.parent.exists():
        return
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    n = 0
    for src in list(PRED_DIR.glob("*.jsonl")) + list(PRED_DIR.glob("metrics.*")) + list(PRED_DIR.glob("coverage.json")):
        if best_effort_copy(src, DRIVE_DIR / src.name):
            n += 1
    print(f"Pushed {n} artifact(s) -> {DRIVE_DIR}", flush=True)


def _reap(pid: int):
    """Return (still_running, exit_code). Reap a finished child so a zombie does not look alive."""
    try:
        waited, status = os.waitpid(pid, os.WNOHANG)
    except ChildProcessError:
        return False, 0
    if waited:
        if os.WIFEXITED(status):
            return False, os.WEXITSTATUS(status)
        return False, 1
    try:
        os.kill(pid, 0)
    except OSError:
        return False, 0
    return True, None


def _pid_alive(pid: int) -> bool:
    running, _code = _reap(pid)
    return running


def _log_finished(log_path: Path, label: str) -> bool:
    mark = "SMOKE OK" if label == "SMOKE" else "FULL OK"
    try:
        text = log_path.read_text(encoding="utf-8", errors="replace")
    except OSError:
        return False
    return mark in text


def _kill_pg(pid: int) -> None:
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(pid, sig)
        except OSError:
            try:
                os.kill(pid, sig)
            except OSError:
                return
        if sig == signal.SIGTERM:
            time.sleep(2)


def _read_job():
    if not JOB_FILE.exists():
        return None
    try:
        job = json.loads(JOB_FILE.read_text(encoding="utf-8"))
        pid = int(job["pid"])
    except Exception:
        return None
    if not _pid_alive(pid):
        JOB_FILE.unlink(missing_ok=True)
        return None
    return job


def _tail_log(log_path: Path, last_size: int) -> int:
    try:
        data = log_path.read_bytes()
    except OSError:
        return last_size
    if len(data) > last_size:
        print(data[last_size:].decode("utf-8", errors="replace"), end="", flush=True)
        return len(data)
    return last_size


def run_detached(label: str, extra: list, log_path: Path) -> int:
    job = _read_job()
    if job:
        print(f"Reattaching to pid={job['pid']} log={job['log']}")
        log_path = Path(job["log"])
        pid = int(job["pid"])
    else:
        log_path.write_text("", encoding="utf-8")
        cmd = base_cmd + extra
        print("$", " ".join(cmd), flush=True)
        proc = subprocess.Popen(
            cmd, cwd=str(ROOT), env=eval_env(),
            stdout=log_path.open("w", encoding="utf-8"),
            stderr=subprocess.STDOUT, start_new_session=True,
        )
        pid = proc.pid
        JOB_FILE.write_text(json.dumps({"pid": pid, "log": str(log_path), "label": label}), encoding="utf-8")
        print(f"started pid={pid} log={log_path}", flush=True)
    last = 0
    rc = 0
    grace_deadline = None
    while True:
        running, code = _reap(pid)
        last = _tail_log(log_path, last)
        if not running:
            rc = 0 if code is None else int(code)
            break
        if _log_finished(log_path, label):
            if grace_deadline is None:
                grace_deadline = time.time() + 20
                print(f"{label} log complete; waiting up to 20s for process exit", flush=True)
            elif time.time() >= grace_deadline:
                print(f"{label} still alive after the log finished; killing pid={pid}", flush=True)
                _kill_pg(pid)
                _running, code = _reap(pid)
                rc = 0 if code is None else int(code)
                break
        time.sleep(5)
    last = _tail_log(log_path, last)
    JOB_FILE.unlink(missing_ok=True)
    push_drive()
    print(f"{label} exit={rc}", flush=True)
    return rc


if SMOKE_FIRST:
    rc = run_detached("SMOKE", ["smoke", "--mode", "both"], ROOT / "eval_nextgqa_smoke.log")
    assert rc == 0, "Smoke failed — xem /content/eval_nextgqa_smoke.log"
    print("SMOKE OK — set SMOKE_FIRST=False rồi chạy full val/oracle.")
if RUN_FULL and not SMOKE_FIRST:
    rc = run_detached("FULL", ["run"], ROOT / "eval_nextgqa_full.log")
    assert rc == 0, "Full run failed — xem /content/eval_nextgqa_full.log"
    print("FULL OK")
if RUN_FULL and SMOKE_FIRST:
    print("RUN_FULL skipped while SMOKE_FIRST=True. Đặt SMOKE_FIRST=False sau khi smoke in timestamp/hash.")


## 8. Report

Bảng accuracy tổng, Acc_C / Acc_T / Acc_D, delta Hybrid−Vanilla. Không gọi số oracle là Acc@GQA.


In [ ]:
# 8. NExT-GQA report
from __future__ import annotations

import subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "dtr_env" / "bin" / "python"
PRED_DIR = ROOT / "outputs_reljoin_v6_videollava" / "eval_nextgqa"
subprocess.run([str(PYTHON), "-u", str(ROOT / "run_nextgqa_reljoin.py"), "report", "--out-dir", str(PRED_DIR)], check=True)
for name in ["coverage.json", "metrics.md", "metrics.json", "predictions.jsonl", "frame_manifest.jsonl"]:
    p = PRED_DIR / name
    print(name, "OK" if p.exists() else "MISSING", p)


## 9. Copy artifacts sang Drive


In [ ]:
# 9. Copy NExT-GQA eval artifacts to Drive
from __future__ import annotations

import shutil
from pathlib import Path

ROOT = Path("/content")
PRED_DIR = ROOT / "outputs_reljoin_v6_videollava" / "eval_nextgqa"
DRIVE = Path("/content/drive/MyDrive/reljoin_v6_videollava_outputs") / "eval_nextgqa"
DRIVE.mkdir(parents=True, exist_ok=True)
n = 0
for src in PRED_DIR.glob("*"):
    if src.is_file():
        shutil.copy2(src, DRIVE / src.name)
        n += 1
        print("copied", src.name)
print({"n_copied": n, "drive": str(DRIVE)})
